# Model B: Two-Class Detection (Parasite + WBC)

This notebook trains YOLOv8 for two-class detection: parasites and white blood cells (WBC).
- **Model**: YOLOv8n (nano)
- **Classes**: 0=parasite, 1=wbc
- **Dataset**: Requires preprocessed annotations with both classes
- **Purpose**: Understand FP distribution relative to WBC locations

In [ ]:
# Cell 1: Mount Google Drive and validate project paths

from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Project root directory
project_root = Path(
    "/content/drive/MyDrive/Spark_2026_Team_Rwanda_Main_Folder/Practicum_Main_Project/"
)

# Dataset directory
dataset_root = project_root / "model_b"

# Dataset YAML file
data_yaml = dataset_root / "data_model_b.yaml"

# Print paths
print(f"Project root: {project_root}")
print(f"Dataset root: {dataset_root}")
print(f"YAML file: {data_yaml}")

# Validate existence
print("\n--- Path Validation ---")
print(f"Project root exists: {project_root.exists()}")
print(f"Dataset root exists: {dataset_root.exists()}")
print(f"YAML file exists: {data_yaml.exists()}")

# Optional detailed checks
if project_root.exists():
    print("\nProject root contents:")
    for item in project_root.iterdir():
        print(f" - {item.name}")

if dataset_root.exists():
    print("\nDataset root contents:")
    for item in dataset_root.iterdir():
        print(f" - {item.name}")

In [ ]:
# Cell 2: Install Ultralytics (if not already installed)
import subprocess, sys, wandb, yaml, os
from collections import Counter

print("Installing Ultralytics...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
print("Ultralytics installed!")

In [ ]:
# Cell 3: Inspect label files and display class IDs with class names

# Load class names from YAML
with open(data_yaml, "r") as f:
    data_config = yaml.safe_load(f)

class_names = data_config["names"]

# Get all training label files
train_labels = list((dataset_root / "train" / "labels").glob("*.txt"))

print(f"Found {len(train_labels)} training label files")

# Detect class IDs present in the dataset
class_counter = Counter()

for label_file in train_labels[:100]:  # Sample first 100 files
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()

            if parts:
                class_id = int(parts[0])
                class_counter[class_id] += 1

# Display detected classes with names
print("\nClasses detected:")

for class_id, count in sorted(class_counter.items()):
    class_name = class_names[class_id]
    print(f"  Class {class_id} ({class_name}): {count} instances")

# Total number of unique detected classes
num_classes = len(class_counter)

print(f"\nTotal unique classes detected: {num_classes}")
print(f"Total classes defined in YAML: {len(class_names)}")

In [ ]:
import os, wandb

api_key = os.environ.get('WANDB_API_KEY')
if not api_key:
    api_key = input('WANDB API key (or set WANDB_API_KEY env var): ').strip()

if api_key:
    os.environ['WANDB_API_KEY'] = api_key
    try:
        wandb.login(key=api_key, relogin=False)
        print('WandB login succeeded')
    except Exception as e:
        print('WandB login failed:', e)
else:
    print('WANDB API key not provided; continuing without WandB logging')

# Initialize a run (Ultralytics will auto-log if wandb is active)
if wandb.run is None:
    try:
        wandb.init(project='Spark_Practicum', name='model_b_training', entity="brain_bytes_rw", reinit=True)
        print('WandB run initialized:', wandb.run.name)
    except Exception as e:
        print('WandB init failed:', e)


In [ ]:
# Cell 5: Train Model B (YOLOv8n, two-class)
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train
print("Starting Model B training (two-class detection)...")
print(f"Using YAML: {yaml_path}")
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=2048,
    batch=32,
    patience=0,
    device=0,
    project='spark_capstone',
    name='model_b_run',
    save=True,
    verbose=True,
    plots=True,
)

print("\nTraining completed!")
# print(f"Best model: {model.trainer.best_model_path}")


In [ ]:
# Cell 6: Validate Model B
print("Validating Model B...")
metrics = model.val()

print(f"\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

In [ ]:
# Cell 7: Run inference on test set
from pathlib import Path

test_images_dir = dataset_root / "test" / "images"
print(f"Test images directory: {test_images_dir}")

if test_images_dir.exists():
    print("\nRunning inference on test set...")
    results = model.predict(
        source=str(test_images_dir),
        save=False,
        conf=0.25,
        verbose=False
    )
    
    print(f"Predictions completed for {len(results)} images")
    
    # Analyze detections by class
    class_counts = {0: 0, 1: 0}
    for r in results:
        for box in r.boxes:
            class_id = int(box.cls[0])
            if class_id in class_counts:
                class_counts[class_id] += 1
    
    print(f"\nDetection summary (conf >= 0.25):")
    print(f"  Parasites (class 0): {class_counts[0]}")
    print(f"  WBCs (class 1): {class_counts[1]}")

In [ ]:
from pathlib import Path
import shutil

best_model_path = Path(model.trainer.best)  # fixed
drive_model_dir = Path("/content/drive/MyDrive/Sam/new_spark_capstone/model_checkpoints")
drive_model_dir.mkdir(exist_ok=True)

dest_path = drive_model_dir / "model_b_best.pt"
shutil.copy(best_model_path, dest_path)

print(f"✅ Model B saved to: {dest_path}")
print(f"File size: {dest_path.stat().st_size / (1024*1024):.2f} MB")